In [1]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import warnings

warnings.filterwarnings('ignore')

In [2]:
def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns


In [3]:
pd.set_option('display.max_columns', 60)

In [4]:
df=pd.read_csv("COTA_KINGElvis_no_aa.csv")
ke_df=pd.read_excel('COTA_KINGElvis.xlsx',sheet_name='Elvis_Review')

In [5]:
df.head()

,Elvis_Date,elvis_id,1st Cleaner,2nd Cleaner,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],SUPERVISOR_COMMENT,id,Completed,INTERV_INIT,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,HAVE_5_MIN_FOR_SURVECode,HAVE_5_MIN_FOR_SURVE,ORIGIN_PLACE_TYPE,ORIGIN_ADDRESS_LAT,ORIGIN_ADDRESS_LONG,PREV_TRANSFERSCode,ORIGIN_TRANSPORTCode,ORIGIN_TRANSPORT,DESTIN_PLACE_TYPE,DESTIN_ADDRESS_LAT,DESTIN_ADDRESS_LONG,NEXT_TRANSFERSCode,DESTIN_TRANSPORTCode,DESTIN_TRANSPORT,STOP_ON_LAT,STOP_ON_LONG,STOP_OFF_LAT,STOP_OFF_LONG,ELVIS_STATUS,ELVIS_COMMENT
0,2023-12-11,5567,,,,,NaN,5567,2023-10-26 16:30:15,999,1,NaN,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-12-11,5568,,,,,NaN,5568,2023-10-26 17:00:33,999,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-12-11,5570,,,,,NaN,5570,2023-10-26 17:01:13,999,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-12-11,5573,,,,,NaN,5573,2023-10-26 17:10:18,999,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,4,No (refused),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-12-11,5575,,,,,NaN,5575,2023-10-26 17:13:04,999,COT_1_34_01,34 MORSE - WEST,5,Do not speak the interviewer's language,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df= df[(df['INTERV_INIT'] != '999') & (df['INTERV_INIT'] != 999)]
df = df[(df['HAVE_5_MIN_FOR_SURVECode'] == 1)]
df.head()

,Elvis_Date,elvis_id,1st Cleaner,2nd Cleaner,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],SUPERVISOR_COMMENT,id,Completed,INTERV_INIT,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,HAVE_5_MIN_FOR_SURVECode,HAVE_5_MIN_FOR_SURVE,ORIGIN_PLACE_TYPE,ORIGIN_ADDRESS_LAT,ORIGIN_ADDRESS_LONG,PREV_TRANSFERSCode,ORIGIN_TRANSPORTCode,ORIGIN_TRANSPORT,DESTIN_PLACE_TYPE,DESTIN_ADDRESS_LAT,DESTIN_ADDRESS_LONG,NEXT_TRANSFERSCode,DESTIN_TRANSPORTCode,DESTIN_TRANSPORT,STOP_ON_LAT,STOP_ON_LONG,STOP_OFF_LAT,STOP_OFF_LONG,ELVIS_STATUS,ELVIS_COMMENT
16,2023-12-11,5632,,,,,NaN,5632,2023-10-30 12:24:56,JTC,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",Your usual workplace,40.014357,-82.986368,1.0,5,Dropped off by someone going elsewhere,Your HOME,NaN,NaN,0.0,1.0,Walk,39.962784,-83.000898,39.943327,-82.853257,NaN,NaN
17,2023-12-11,5634,,,,,NaN,5634,2023-10-30 12:30:07,BLN,COT_1_2_00,2 E MAIN/N HIGH - SOUTH/EAST,1,"<span style=""color:#000080;""><span style=""font...",Your HOME,NaN,NaN,0.0,1,Walk,Grocery / Food Shopping,39.889314,-83.001389,1.0,1.0,Walk,39.962784,-83.000898,39.956175,-82.998467,NaN,NaN
18,2023-12-11,5636,,,,,NaN,5636,2023-10-30 12:36:28,JTC,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",Social Visit (friends/relatives),39.989274,-83.005284,0.0,1,Walk,Your HOME,NaN,NaN,0.0,1.0,Walk,39.988958,-83.005951,39.944646,-82.876777,NaN,NaN
19,2023-12-11,5639,,,,,NaN,5639,2023-10-30 12:46:45,MCB,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",College / University (students only),39.968780,-82.986813,2.0,1,Walk,Your HOME,NaN,NaN,0.0,1.0,Walk,39.968203,-83.001929,39.965149,-83.001318,Delete,NaN
24,2023-12-11,5646,,,,,o= home,5646,2023-10-30 12:44:33,CDK,COT_1_2_00,2 E MAIN/N HIGH - SOUTH/EAST,1,"<span style=""color:#000080;""><span style=""font...","Personal Business (bank, haircut, post office)",39.955255,-82.881798,0.0,1,Walk,"Personal Business (bank, haircut, post office)",39.955255,-82.881798,0.0,1.0,Walk,39.957720,-82.967026,39.955554,-82.882520,Fixable - needs additional review,o= home


In [7]:
df.shape

(4135, 32)

In [8]:
ke_df = ke_df[(ke_df['INTERV_INIT'] != '999') & (ke_df['INTERV_INIT'] != 999)]
ke_df = ke_df[(ke_df['HAVE_5_MIN_FOR_SURVECode'] == 1)]
ke_df.head()

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,id,Completed,INTERV_INIT,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,HAVE_5_MIN_FOR_SURVECode,HAVE_5_MIN_FOR_SURVE,ORIGIN_PLACE_TYPE,ORIGIN_TRANSPORTCode,ORIGIN_TRANSPORT,DESTIN_PLACE_TYPE,DESTIN_TRANSPORTCode,DESTIN_TRANSPORT,_INTRV_NOTE,ELVIS_STATUS,ELVIS_COMMENT,ROUTE_STATUS,Stops_Status,Test_Status,Traditional_Check,OD_Distance_Check,Transfer_Distance_Check,2X_REVIEW_CHECK,2X_REVIEW_CHECK.1,2x_REVIEWED_BY,2x_REVIEWED_FLAG,ADMIN_APPROVED,SURVEY_RECOVERY,SURVEY_RECOVERY_REVIEWED_BY,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1
16,2023-11-03,5632,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5632,2023-10-30 12:24:56,JTC,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",Your usual workplace,5,Dropped off by someone going elsewhere,Your HOME,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
17,2023-11-03,5634,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5634,2023-10-30 12:30:07,BLN,COT_1_2_00,2 E MAIN/N HIGH - SOUTH/EAST,1,"<span style=""color:#000080;""><span style=""font...",Your HOME,1,Walk,Grocery / Food Shopping,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
18,2023-11-03,5636,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5636,2023-10-30 12:36:28,JTC,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",Social Visit (friends/relatives),1,Walk,Your HOME,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
19,2023-11-03,5639,HereAPI,"T A Divya, Brian, Regina",Remove,Round trip,,Elvis_Review,Elvis_Review,NaN,5639,2023-10-30 12:46:45,MCB,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",College / University (students only),1,Walk,Your HOME,1,Walk,NaN,Delete,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
24,2023-11-03,5646,HereAPI,"T A Divya, Brian, Regina",Use,NaN,,Elvis_Review,Elvis_Review,"o= home = E Main St & Wilson Ave, Columbus, OH...",5646,2023-10-30 12:44:33,CDK,COT_1_2_00,2 E MAIN/N HIGH - SOUTH/EAST,1,"<span style=""color:#000080;""><span style=""font...","Personal Business (bank, haircut, post office)",1,Walk,"Personal Business (bank, haircut, post office)",1,Walk,NaN,Fixable - needs additional review,o= home,Review,Review,Tested,0,1,0,1,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [9]:
ke_df.shape

(4135, 43)

In [10]:
ke_df.columns

Index(['Elvis_Date', 'elvis_id', '1st Cleaner', 'FINAL_REVIEWER',
       'Final_Usage', 'REASON FOR REMOVAL', 'REASON FOR REMOVAL [Other]',
       'route_match_flag', 'distance_flag', 'SUPERVISOR_COMMENT', 'id',
       'Completed', 'INTERV_INIT', 'ROUTE_SURVEYEDCode', 'ROUTE_SURVEYED',
       'HAVE_5_MIN_FOR_SURVECode', 'HAVE_5_MIN_FOR_SURVE', 'ORIGIN_PLACE_TYPE',
       'ORIGIN_TRANSPORTCode', 'ORIGIN_TRANSPORT', 'DESTIN_PLACE_TYPE',
       'DESTIN_TRANSPORTCode', 'DESTIN_TRANSPORT', '_INTRV_NOTE',
       'ELVIS_STATUS', 'ELVIS_COMMENT', 'ROUTE_STATUS', 'Stops_Status',
       'Test_Status', 'Traditional_Check', 'OD_Distance_Check',
       'Transfer_Distance_Check', '2X_REVIEW_CHECK', '2X_REVIEW_CHECK.1',
       '2x_REVIEWED_BY', '2x_REVIEWED_FLAG', 'ADMIN_APPROVED',
       'SURVEY_RECOVERY', 'SURVEY_RECOVERY_REVIEWED_BY', 'Recovery Check',
       '2x_REVIEWED_BY.1', '2x_REVIEWED_FLAG.1', 'ADMIN_APPROVED.1'],
      dtype='object')

In [11]:
df=pd.merge(df,ke_df[['id','Final_Usage','FINAL_REVIEWER','Traditional_Check', 'OD_Distance_Check',
       'Transfer_Distance_Check', '2X_REVIEW_CHECK', '2X_REVIEW_CHECK.1',
       '2x_REVIEWED_BY', '2x_REVIEWED_FLAG', 'ADMIN_APPROVED',
       'SURVEY_RECOVERY', 'SURVEY_RECOVERY_REVIEWED_BY', 'Recovery Check',
       '2x_REVIEWED_BY.1', '2x_REVIEWED_FLAG.1', 'ADMIN_APPROVED.1']],on='id',how='left')

In [12]:
df = df[(df['INTERV_INIT'] != '999') & (df['INTERV_INIT'] != 999)]

In [13]:
df.to_csv("latest_COTA_KINGElvis.csv",index=False)

# Review The Reviewer Stats New Logic

In [14]:
ke_df=pd.read_excel("COTA_KINGElvis.xlsx",sheet_name='Elvis_Review')
ke_df=ke_df[ke_df['INTERV_INIT']!=999]
ke_df=ke_df[ke_df['HAVE_5_MIN_FOR_SURVECode']==1]
ke_df

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,id,Completed,INTERV_INIT,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,HAVE_5_MIN_FOR_SURVECode,HAVE_5_MIN_FOR_SURVE,ORIGIN_PLACE_TYPE,ORIGIN_TRANSPORTCode,ORIGIN_TRANSPORT,DESTIN_PLACE_TYPE,DESTIN_TRANSPORTCode,DESTIN_TRANSPORT,_INTRV_NOTE,ELVIS_STATUS,ELVIS_COMMENT,ROUTE_STATUS,Stops_Status,Test_Status,Traditional_Check,OD_Distance_Check,Transfer_Distance_Check,2X_REVIEW_CHECK,2X_REVIEW_CHECK.1,2x_REVIEWED_BY,2x_REVIEWED_FLAG,ADMIN_APPROVED,SURVEY_RECOVERY,SURVEY_RECOVERY_REVIEWED_BY,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1
16,2023-11-03,5632,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5632,2023-10-30 12:24:56,JTC,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",Your usual workplace,5,Dropped off by someone going elsewhere,Your HOME,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
17,2023-11-03,5634,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5634,2023-10-30 12:30:07,BLN,COT_1_2_00,2 E MAIN/N HIGH - SOUTH/EAST,1,"<span style=""color:#000080;""><span style=""font...",Your HOME,1,Walk,Grocery / Food Shopping,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
18,2023-11-03,5636,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5636,2023-10-30 12:36:28,JTC,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",Social Visit (friends/relatives),1,Walk,Your HOME,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
19,2023-11-03,5639,HereAPI,"T A Divya, Brian, Regina",Remove,Round trip,,Elvis_Review,Elvis_Review,NaN,5639,2023-10-30 12:46:45,MCB,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",College / University (students only),1,Walk,Your HOME,1,Walk,NaN,Delete,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
24,2023-11-03,5646,HereAPI,"T A Divya, Brian, Regina",Use,NaN,,Elvis_Review,Elvis_Review,"o= home = E Main St & Wilson Ave, Columbus, OH...",5646,2023-10-30 12:44:33,CDK,COT_1_2_00,2 E MAIN/N HIGH - SOUTH/EAST,1,"<span style=""color:#000080;""><span style=""font...","Personal Business (bank, haircut, post office)",1,Walk,"Personal Business (bank, haircut, post office)",1,Walk,NaN,Fixable - needs additional review,o= home,Review,Review,Tested,0,1,0,1,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5566,2023-12-03,13637,HereAPI,Divya sravani,Use,NaN,,Elvis_Review,Elvis_Review,NaN,13637,2023-12-01 12:08:29,ANC,COT_1_2_00,2 E MAIN/N HIGH - SOUTH/EAST,1,"<span style=""color:#000080;""><span style=""font...",Other shopping,1,Walk,Social Visit (friends/relatives),1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5568,2023-12-03,13642,HereAPI,Divya sravani,Use,NaN,,Elvis_Review,Elvis_Review,NaN,13642,2023-12-01 15:02:36,ANC,COT_1_9_00,9 W MOUND/BRENTNELL - NORTH,1,"<span style=""color:#000080;""><span style=""font...",Your usual workplace,1,Walk,Your HOME,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5569,2023-12-03,13643,HereAPI,Divya sravani,Use,NaN,,Elvis_Review,Elvis_Review,NaN,13643,2023-12-01 15:33:31,ANC,COT_1_9_00,9 W MOUND/BRENTNELL - NORTH,1,"<span style=""color:#000080;""><span style=""font...",Social Visit (friends/relatives),1,Walk,Your HOME,5,Be picked up by someone,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5573,2023-12-03,13647,HereAPI,Divya sravani,Use,NaN,,Elvis_Review,Elvis_Review,NaN,13647,2023-12-01 16:52:58,ANC,COT_1_25_00,25

In [15]:
df=pd.read_csv('elviscota2023obweekday_export_odbc.csv')
df.head()

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,EXTRATRIP_ID,EXTRATRIP_MAIN_ID,COMPANION_ID,COMPANION_MAIN_ID,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,ROUTE_SURVEYED_Other,RANDOM_NUMBER,HAVE_5_MIN_FOR_SURVECode,HAVE_5_MIN_FOR_SURVE,HV5MIN_TIME,SENDLINK_NAME,SENDLINK_EMAIL,SENDLINK_PHONE,SENDLINK_NOTE,SELECT_LANGUAGECode,SELECT_LANGUAGE,SELECT_LANGUAGE_Other,SPANISH_PAPER_CALLCode,SPANISH_PAPER_CALL,...,ELLVIS_EXTRA1,ELVIS_HISTORY,ELVIS_USER_CHANGE_1_C_DATE,ELVIS_USER_CHANGE_1_C_FIELD,ELVIS_USER_CHANGE_1_C_REASON,ELVIS_USER_CHANGE_2_C_DATE,ELVIS_USER_CHANGE_2_C_FIELD,ELVIS_USER_CHANGE_2_C_REASON,ELVIS_USER_CHANGE_3_C_DATE,ELVIS_USER_CHANGE_3_C_FIELD,ELVIS_USER_CHANGE_3_C_REASON,ELVIS_USER_CHANGE_4_C_DATE,ELVIS_USER_CHANGE_4_C_FIELD,ELVIS_USER_CHANGE_4_C_REASON,ELVIS_USER_CHANGE_5_C_DATE,ELVIS_USER_CHANGE_5_C_FIELD,ELVIS_USER_CHANGE_5_C_REASON,ELVIS_USER_CHANGE_6_C_DATE,ELVIS_USER_CHANGE_6_C_FIELD,ELVIS_USER_CHANGE_6_C_REASON,ELVIS_USER_CHANGE_7_C_DATE,ELVIS_USER_CHANGE_7_C_FIELD,ELVIS_USER_CHANGE_7_C_REASON,INVITE_EMAIL,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,_REVERSE_TRIP,GENERATABLE_TRIPS
0,5567,2023-10-26 16:30:15,78,en,2023-10-26 16:29:55,2023-10-26 16:30:15,70.113.126.123,https://cota-oh.etc-research.com/,-14400,999,NaN,NaN,NaN,NaN,1,NaN,NaN,1,4,No (refused),2023-10-26 16:30:11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5568,2023-10-26 17:00:33,78,en,2023-10-26 17:00:12,2023-10-26 17:00:33,70.190.112.58,https://cota-oh.etc-research.com/index.php/adm...,-14400,999,NaN,NaN,NaN,NaN,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,NaN,1,4,No (refused),2023-10-26 17:00:26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5570,2023-10-26 17:01:13,78,en,2023-10-26 17:00:57,2023-10-26 17:01:13,70.190.112.58,https://cota-oh.etc-research.com/index.php/adm...,-14400,999,NaN,NaN,NaN,NaN,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,NaN,6,4,No (refused),2023-10-26 17:01:05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5573,2023-10-26 17:10:18,78,en,2023-10-26 17:04:26,2023-10-26 17:10:18,70.113.126.123,NaN,-14400,999,NaN,NaN,NaN,NaN,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,NaN,3,4,No (refused),2023-10-26 17:10:14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5575,2023-10-26 17:13:04,78,en,2023-10-26 17:10:22,2023-10-26 17:13:04,70.113.126.123,https://cota-oh.etc-research.com/index.php/sur...,-14400,999,NaN,NaN,NaN,NaN,COT_1_34_01,34 MORSE - WEST,NaN,3,5,Do not speak the interviewer's language,2023-10-26 17:12:20,NaN,NaN,NaN,NaN,CANT,å»£æ±è©± (CHINESE - CANTONESE),NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
cota_df=pd.read_csv('cota2023obweekday_export_odbc.csv')
cota_df.head()

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,EXTRATRIP_ID,EXTRATRIP_MAIN_ID,COMPANION_ID,COMPANION_MAIN_ID,ROUTE_SURVEYED_Code_,ROUTE_SURVEYED,ROUTE_SURVEYED_Other_,RANDOM_NUMBER,HAVE_5_MIN_FOR_SURVE_Code_,HAVE_5_MIN_FOR_SURVE,HV5MIN_TIME,SENDLINK_NAME_,SENDLINK_EMAIL_,SENDLINK_PHONE_,SENDLINK_NOTE_,SELECT_LANGUAGE_Code_,SELECT_LANGUAGE,SELECT_LANGUAGE_Other_,SPANISH_PAPER_CALL_Code_,SPANISH_PAPER_CALL,...,ELLVIS_EXTRA1,ELVIS_HISTORY,ELVIS_USER_CHANGE_1_C_DATE_,ELVIS_USER_CHANGE_1_C_FIELD_,ELVIS_USER_CHANGE_1_C_REASON_,ELVIS_USER_CHANGE_2_C_DATE_,ELVIS_USER_CHANGE_2_C_FIELD_,ELVIS_USER_CHANGE_2_C_REASON_,ELVIS_USER_CHANGE_3_C_DATE_,ELVIS_USER_CHANGE_3_C_FIELD_,ELVIS_USER_CHANGE_3_C_REASON_,ELVIS_USER_CHANGE_4_C_DATE_,ELVIS_USER_CHANGE_4_C_FIELD_,ELVIS_USER_CHANGE_4_C_REASON_,ELVIS_USER_CHANGE_5_C_DATE_,ELVIS_USER_CHANGE_5_C_FIELD_,ELVIS_USER_CHANGE_5_C_REASON_,ELVIS_USER_CHANGE_6_C_DATE_,ELVIS_USER_CHANGE_6_C_FIELD_,ELVIS_USER_CHANGE_6_C_REASON_,ELVIS_USER_CHANGE_7_C_DATE_,ELVIS_USER_CHANGE_7_C_FIELD_,ELVIS_USER_CHANGE_7_C_REASON_,INVITE_EMAIL,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,_REVERSE_TRIP,GENERATABLE_TRIPS
0,5567,2023-10-26 16:30:15,78,en,2023-10-26 16:29:55,2023-10-26 16:30:15,70.113.126.123,https://cota-oh.etc-research.com/,-14400,999,NaN,NaN,NaN,NaN,1,NaN,NaN,1,4,No (refused),2023-10-26 16:30:11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5568,2023-10-26 17:00:33,78,en,2023-10-26 17:00:12,2023-10-26 17:00:33,70.190.112.58,https://cota-oh.etc-research.com/index.php/adm...,-14400,999,NaN,NaN,NaN,NaN,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,NaN,1,4,No (refused),2023-10-26 17:00:26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5570,2023-10-26 17:01:13,78,en,2023-10-26 17:00:57,2023-10-26 17:01:13,70.190.112.58,https://cota-oh.etc-research.com/index.php/adm...,-14400,999,NaN,NaN,NaN,NaN,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,NaN,6,4,No (refused),2023-10-26 17:01:05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5573,2023-10-26 17:10:18,78,en,2023-10-26 17:04:26,2023-10-26 17:10:18,70.113.126.123,NaN,-14400,999,NaN,NaN,NaN,NaN,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,NaN,3,4,No (refused),2023-10-26 17:10:14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5575,2023-10-26 17:13:04,78,en,2023-10-26 17:10:22,2023-10-26 17:13:04,70.113.126.123,https://cota-oh.etc-research.com/index.php/sur...,-14400,999,NaN,NaN,NaN,NaN,COT_1_34_01,34 MORSE - WEST,NaN,3,5,Do not speak the interviewer's language,2023-10-26 17:12:20,NaN,NaN,NaN,NaN,CANT,å»£æ±è©± (CHINESE - CANTONESE),NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
df.shape

(5581, 745)

In [18]:
ke_df['id']

16       5632
17       5634
18       5636
19       5639
24       5646
        ...  
5566    13637
5568    13642
5569    13643
5573    13647
5575    13650
Name: id, Length: 4135, dtype: int64

In [19]:
df=pd.merge(df,ke_df[['id']],on='id',how='left',indicator=True)
df['Matched'] = (df['_merge'] == 'both').astype(int)
df.drop(columns=['_merge'])
df=df[df['Matched']==1]
df.drop_duplicates(subset=['id'],keep='first',inplace=True)

In [20]:
cota_df=pd.merge(cota_df,ke_df[['id']],on='id',how='left',indicator=True)
cota_df['Matched'] = (cota_df['_merge'] == 'both').astype(int)
cota_df.drop(columns=['_merge'])
cota_df=cota_df[cota_df['Matched']==1]
cota_df.drop_duplicates(subset=['id'],keep='first',inplace=True)

In [21]:
cota_df.shape

(4135, 747)

In [22]:
df.shape

(4135, 747)

changes for Origin, Destination, prev transfer, next transfer, etc... 

In [23]:
origin_destin_prev_next_columns_check=['originaddresslat', 'originaddresslong', 'stoponlat', 'stoponlong', 'stopofflat', 'stopofflong','destinaddresslat', 'destinaddresslong','prevtransferscode','nexttransferscode']
origin_destin_prev_next_columns=check_all_characters_present(df,origin_destin_prev_next_columns_check)
origin_destin_prev_next_columns.sort()
origin_destin_prev_next_columns

['DESTIN_ADDRESS_LAT',
 'DESTIN_ADDRESS_LONG',
 'NEXT_TRANSFERSCode',
 'ORIGIN_ADDRESS_LAT',
 'ORIGIN_ADDRESS_LONG',
 'PREV_TRANSFERSCode',
 'STOP_OFF_LAT',
 'STOP_OFF_LONG',
 'STOP_ON_LAT',
 'STOP_ON_LONG']

In [24]:
origin_home_columns_check=['originaddresslat', 'originaddresslong','originplacetype','homeaddresslat','homeaddresslong']
origin_home_columns=check_all_characters_present(df,origin_home_columns_check)
origin_home_columns.sort()
origin_home_columns

['HOME_ADDRESS_LAT',
 'HOME_ADDRESS_LONG',
 'ORIGIN_ADDRESS_LAT',
 'ORIGIN_ADDRESS_LONG',
 'ORIGIN_PLACE_TYPE']

In [25]:
destin_home_columns_check=['destinaddresslat', 'destinaddresslong','destinplacetype','homeaddresslat','homeaddresslong']
destin_home_columns=check_all_characters_present(df,destin_home_columns_check)
destin_home_columns.sort()
destin_home_columns

['DESTIN_ADDRESS_LAT',
 'DESTIN_ADDRESS_LONG',
 'DESTIN_PLACE_TYPE',
 'HOME_ADDRESS_LAT',
 'HOME_ADDRESS_LONG']

In [26]:
df[destin_home_columns][(df['id']==5636)|(df['id']==5632)|(df['id']==5639)].head()

,DESTIN_ADDRESS_LAT,DESTIN_ADDRESS_LONG,DESTIN_PLACE_TYPE,HOME_ADDRESS_LAT,HOME_ADDRESS_LONG
16,NaN,NaN,Your HOME,39.941384,-82.853440
18,NaN,NaN,Your HOME,39.944775,-82.877817
19,NaN,NaN,Your HOME,39.968780,-82.986813


In [27]:
data='Your Work'
if 'home' in data.lower():
    print("Home Present")

In [28]:
for index, row in df.iterrows():
    if (pd.isna(row[origin_home_columns[2]]) or pd.isna(row[origin_home_columns[3]])) and 'home' in row[origin_home_columns[4]].lower():
        df.loc[index, origin_home_columns[2]] = row[origin_home_columns[0]]
        df.loc[index, origin_home_columns[3]] = row[origin_home_columns[1]]
        
    if (pd.isna(row[destin_home_columns[0]]) or pd.isna(row[destin_home_columns[1]])) and 'home' in str(row[destin_home_columns[2]]).lower():
        df.loc[index, destin_home_columns[0]] = row[destin_home_columns[3]]
        df.loc[index, destin_home_columns[1]] = row[destin_home_columns[4]]

In [29]:
df[destin_home_columns][(df['id']==5636)|(df['id']==5632)|(df['id']==5639)].head()

,DESTIN_ADDRESS_LAT,DESTIN_ADDRESS_LONG,DESTIN_PLACE_TYPE,HOME_ADDRESS_LAT,HOME_ADDRESS_LONG
16,39.941384,-82.853440,Your HOME,39.941384,-82.853440
18,39.944775,-82.877817,Your HOME,39.944775,-82.877817
19,39.968780,-82.986813,Your HOME,39.968780,-82.986813


In [30]:
non_origin_home_columns_check=['originaddresslat', 'originaddresslong','originplacetype','homeaddresslat','homeaddresslong']
non_origin_home_columns=check_all_characters_present(cota_df,non_origin_home_columns_check)
non_origin_home_columns.sort()
non_origin_home_columns

['HOME_ADDRESS_LAT_',
 'HOME_ADDRESS_LONG_',
 'ORIGIN_ADDRESS_LAT_',
 'ORIGIN_ADDRESS_LONG_',
 'ORIGIN_PLACE_TYPE']

In [31]:
non_destin_home_columns_check=['destinaddresslat', 'destinaddresslong','destinplacetype','homeaddresslat','homeaddresslong']
non_destin_home_columns=check_all_characters_present(cota_df,non_destin_home_columns_check)
non_destin_home_columns.sort()
non_destin_home_columns

['DESTIN_ADDRESS_LAT_',
 'DESTIN_ADDRESS_LONG_',
 'DESTIN_PLACE_TYPE',
 'HOME_ADDRESS_LAT_',
 'HOME_ADDRESS_LONG_']

In [32]:
cota_df[non_origin_home_columns][(cota_df['id']==5634)|(cota_df['id']==5632)].head()

,HOME_ADDRESS_LAT_,HOME_ADDRESS_LONG_,ORIGIN_ADDRESS_LAT_,ORIGIN_ADDRESS_LONG_,ORIGIN_PLACE_TYPE
16,39.941384,-82.853440,40.014357,-82.986368,Your usual workplace
17,39.963826,-82.999827,NaN,NaN,Your HOME


In [33]:
cota_df[non_destin_home_columns][(cota_df['id']==6309)|(cota_df['id']==6334)|(cota_df['id']==6338)].head()

,DESTIN_ADDRESS_LAT_,DESTIN_ADDRESS_LONG_,DESTIN_PLACE_TYPE,HOME_ADDRESS_LAT_,HOME_ADDRESS_LONG_
525,NaN,NaN,Your HOME,39.960728,-83.014572
543,NaN,NaN,Your HOME,39.960282,-82.789292
546,NaN,NaN,Your HOME,39.947258,-82.968260


In [34]:
for index, row in cota_df.iterrows():
    if (pd.isna(row[non_origin_home_columns[2]]) or pd.isna(row[non_origin_home_columns[3]])) and ('home' in row[non_origin_home_columns[4]].lower()) :
        cota_df.loc[index, non_origin_home_columns[2]] = row[non_origin_home_columns[0]]
        cota_df.loc[index, non_origin_home_columns[3]] = row[non_origin_home_columns[1]]
    if (pd.isna(row[non_destin_home_columns[0]]) or pd.isna(row[non_destin_home_columns[1]])) and ('home' in str(row[non_destin_home_columns[2]]).lower()) :
        cota_df.loc[index, non_destin_home_columns[0]] = row[non_destin_home_columns[3]]
        cota_df.loc[index, non_destin_home_columns[1]] = row[non_destin_home_columns[4]]


In [35]:
cota_df[non_destin_home_columns][(cota_df['id']==6309)|(cota_df['id']==6334)|(cota_df['id']==6338)].head()

,DESTIN_ADDRESS_LAT_,DESTIN_ADDRESS_LONG_,DESTIN_PLACE_TYPE,HOME_ADDRESS_LAT_,HOME_ADDRESS_LONG_
525,39.960728,-83.014572,Your HOME,39.960728,-83.014572
543,39.960282,-82.789292,Your HOME,39.960282,-82.789292
546,39.947258,-82.968260,Your HOME,39.947258,-82.968260


In [36]:
cota_df[non_origin_home_columns][(cota_df['id']==5634)|(cota_df['id']==5632)].head()

,HOME_ADDRESS_LAT_,HOME_ADDRESS_LONG_,ORIGIN_ADDRESS_LAT_,ORIGIN_ADDRESS_LONG_,ORIGIN_PLACE_TYPE
16,39.941384,-82.853440,40.014357,-82.986368,Your usual workplace
17,39.963826,-82.999827,39.963826,-82.999827,Your HOME


In [37]:
non_origin_destin_prev_next_columns_check=['originaddresslat', 'originaddresslong', 'stoponlat', 'stoponlong', 'stopofflat', 'stopofflong','destinaddresslat', 'destinaddresslong','prevtransferscode','nexttransferscode']
non_origin_destin_prev_next_columns=check_all_characters_present(cota_df,non_origin_destin_prev_next_columns_check)
non_origin_destin_prev_next_columns.sort()
non_origin_destin_prev_next_columns

['DESTIN_ADDRESS_LAT_',
 'DESTIN_ADDRESS_LONG_',
 'NEXT_TRANSFERS_Code_',
 'ORIGIN_ADDRESS_LAT_',
 'ORIGIN_ADDRESS_LONG_',
 'PREV_TRANSFERS_Code_',
 'STOP_OFF_LAT_',
 'STOP_OFF_LONG_',
 'STOP_ON_LAT_',
 'STOP_ON_LONG_']

In [38]:
# ke_df[[*origin_destin_prev_next_columns]]=df[[*origin_destin_prev_next_columns]]

In [39]:
ke_df=pd.merge(ke_df,df[['id',*origin_destin_prev_next_columns]],on='id',how='left')

In [40]:
ke_df.rename(columns={'DESTIN_ADDRESS_LAT':'KE_DESTIN_ADDRESS_LAT','DESTIN_ADDRESS_LONG':'KE_DESTIN_ADDRESS_LONG',
                     'ORIGIN_ADDRESS_LAT':'KE_ORIGIN_ADDRESS_LAT','ORIGIN_ADDRESS_LONG':'KE_ORIGIN_ADDRESS_LONG',
                      'STOP_OFF_LAT':'KE_STOP_OFF_LAT','STOP_OFF_LONG':'KE_STOP_OFF_LONG','STOP_ON_LAT':'KE_STOP_ON_LAT',
                     'STOP_ON_LONG':'KE_STOP_ON_LONG','PREV_TRANSFERSCode':'KE_PREV_TRANSFERSCode','NEXT_TRANSFERSCode':'KE_NEXT_TRANSFERSCode'
                     },inplace=True)



In [41]:
ke_df.head()

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,id,Completed,INTERV_INIT,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,HAVE_5_MIN_FOR_SURVECode,HAVE_5_MIN_FOR_SURVE,ORIGIN_PLACE_TYPE,ORIGIN_TRANSPORTCode,ORIGIN_TRANSPORT,DESTIN_PLACE_TYPE,DESTIN_TRANSPORTCode,DESTIN_TRANSPORT,_INTRV_NOTE,ELVIS_STATUS,ELVIS_COMMENT,ROUTE_STATUS,Stops_Status,Test_Status,Traditional_Check,OD_Distance_Check,Transfer_Distance_Check,2X_REVIEW_CHECK,2X_REVIEW_CHECK.1,2x_REVIEWED_BY,2x_REVIEWED_FLAG,ADMIN_APPROVED,SURVEY_RECOVERY,SURVEY_RECOVERY_REVIEWED_BY,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1,KE_DESTIN_ADDRESS_LAT,KE_DESTIN_ADDRESS_LONG,KE_NEXT_TRANSFERSCode,KE_ORIGIN_ADDRESS_LAT,KE_ORIGIN_ADDRESS_LONG,KE_PREV_TRANSFERSCode,KE_STOP_OFF_LAT,KE_STOP_OFF_LONG,KE_STOP_ON_LAT,KE_STOP_ON_LONG
0,2023-11-03,5632,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5632,2023-10-30 12:24:56,JTC,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",Your usual workplace,5,Dropped off by someone going elsewhere,Your HOME,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,39.941384,-82.853440,0.0,40.014357,-82.986368,1.0,39.943327,-82.853257,39.962784,-83.000898
1,2023-11-03,5634,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5634,2023-10-30 12:30:07,BLN,COT_1_2_00,2 E MAIN/N HIGH - SOUTH/EAST,1,"<span style=""color:#000080;""><span style=""font...",Your HOME,1,Walk,Grocery / Food Shopping,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,39.889314,-83.001389,1.0,39.963826,-82.999827,0.0,39.956175,-82.998467,39.962784,-83.000898
2,2023-11-03,5636,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5636,2023-10-30 12:36:28,JTC,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",Social Visit (friends/relatives),1,Walk,Your HOME,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,39.944775,-82.877817,0.0,39.989274,-83.005284,0.0,39.944646,-82.876777,39.988958,-83.005951
3,2023-11-03,5639,HereAPI,"T A Divya, Brian, Regina",Remove,Round trip,,Elvis_Review,Elvis_Review,NaN,5639,2023-10-30 12:46:45,MCB,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",College / University (students only),1,Walk,Your HOME,1,Walk,NaN,Delete,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,39.968780,-82.986813,0.0,39.968780,-82.986813,2.0,39.965149,-83.001318,39.968203,-83.001929
4,2023-11-03,5646,HereAPI,"T A Divya, Brian, Regina",Use,NaN,,Elvis_Review,Elvis_Review,"o= home = E Main St & Wilson Ave, Columbus, OH...",5646,2023-10-30 12:44:33,CDK,COT_1_2_00,2 E MAIN/N HIGH - SOUTH/EAST,1,"<span style=""color:#000080;""><span style=""font...","Personal Business (bank, haircut, post office)",1,Walk,"Personal Business (bank, haircut, post office)",1,Walk,NaN,Fixable - needs additional review,o= home,Review,Review,Tested,0,1,0,1,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,39.955255,-82.881798,0.0,39.955255,-82.881798,0.0,39.955554,-82.882520,39.957720,-82.967026


In [42]:
ke_df=pd.merge(ke_df,cota_df[['id',*non_origin_destin_prev_next_columns]],on='id',how='left')

In [43]:
# ke_df=pd.merge(ke_df,df[['id',*origin_destin_prev_next_columns]],on='id',how='left')

In [44]:
ke_df.columns

Index(['Elvis_Date', 'elvis_id', '1st Cleaner', 'FINAL_REVIEWER',
       'Final_Usage', 'REASON FOR REMOVAL', 'REASON FOR REMOVAL [Other]',
       'route_match_flag', 'distance_flag', 'SUPERVISOR_COMMENT', 'id',
       'Completed', 'INTERV_INIT', 'ROUTE_SURVEYEDCode', 'ROUTE_SURVEYED',
       'HAVE_5_MIN_FOR_SURVECode', 'HAVE_5_MIN_FOR_SURVE', 'ORIGIN_PLACE_TYPE',
       'ORIGIN_TRANSPORTCode', 'ORIGIN_TRANSPORT', 'DESTIN_PLACE_TYPE',
       'DESTIN_TRANSPORTCode', 'DESTIN_TRANSPORT', '_INTRV_NOTE',
       'ELVIS_STATUS', 'ELVIS_COMMENT', 'ROUTE_STATUS', 'Stops_Status',
       'Test_Status', 'Traditional_Check', 'OD_Distance_Check',
       'Transfer_Distance_Check', '2X_REVIEW_CHECK', '2X_REVIEW_CHECK.1',
       '2x_REVIEWED_BY', '2x_REVIEWED_FLAG', 'ADMIN_APPROVED',
       'SURVEY_RECOVERY', 'SURVEY_RECOVERY_REVIEWED_BY', 'Recovery Check',
       '2x_REVIEWED_BY.1', '2x_REVIEWED_FLAG.1', 'ADMIN_APPROVED.1',
       'KE_DESTIN_ADDRESS_LAT', 'KE_DESTIN_ADDRESS_LONG',
       'KE_NEXT_TRAN

In [45]:
cota_df[['id',*non_origin_destin_prev_next_columns]]

,id,DESTIN_ADDRESS_LAT_,DESTIN_ADDRESS_LONG_,NEXT_TRANSFERS_Code_,ORIGIN_ADDRESS_LAT_,ORIGIN_ADDRESS_LONG_,PREV_TRANSFERS_Code_,STOP_OFF_LAT_,STOP_OFF_LONG_,STOP_ON_LAT_,STOP_ON_LONG_
16,5632,39.941384,-82.853440,0.0,40.014357,-82.986368,1.0,39.943327,-82.853257,39.962784,-83.000898
17,5634,39.889314,-83.001389,1.0,39.963826,-82.999827,0.0,39.956175,-82.998467,39.962784,-83.000898
18,5636,39.944775,-82.877817,0.0,39.989274,-83.005284,0.0,39.944646,-82.876777,39.988958,-83.005951
19,5639,39.968780,-82.986813,0.0,39.968780,-82.986813,2.0,39.965149,-83.001318,39.968203,-83.001929
24,5646,39.955255,-82.881798,0.0,39.955255,-82.881798,0.0,39.955554,-82.882520,39.957720,-82.967026
...,...,...,...,...,...,...,...,...,...,...,...
5566,13637,39.975582,-83.012655,0.0,39.995013,-83.007298,0.0,39.978411,-83.003898,39.995886,-83.007426
5568,13642,40.008562,-82.935728,0.0,39.969387,-83.004262,0.0,40.010357,-82.947846,39.968768,-83.003008
5569,13643,40.128400,-82.993658,1.0,40.024691,-82.932885,0.0,40.060719,-82.909141,40.027414,-82.933207
5573,13647,40.004199,-82.786555,1.0,39.858534,-82.833692,0.0,39.980259,-82.837110,39.860402,-82.825778


In [46]:
ke_df[['id',*non_origin_destin_prev_next_columns]]

,id,DESTIN_ADDRESS_LAT_,DESTIN_ADDRESS_LONG_,NEXT_TRANSFERS_Code_,ORIGIN_ADDRESS_LAT_,ORIGIN_ADDRESS_LONG_,PREV_TRANSFERS_Code_,STOP_OFF_LAT_,STOP_OFF_LONG_,STOP_ON_LAT_,STOP_ON_LONG_
0,5632,39.941384,-82.853440,0.0,40.014357,-82.986368,1.0,39.943327,-82.853257,39.962784,-83.000898
1,5634,39.889314,-83.001389,1.0,39.963826,-82.999827,0.0,39.956175,-82.998467,39.962784,-83.000898
2,5636,39.944775,-82.877817,0.0,39.989274,-83.005284,0.0,39.944646,-82.876777,39.988958,-83.005951
3,5639,39.968780,-82.986813,0.0,39.968780,-82.986813,2.0,39.965149,-83.001318,39.968203,-83.001929
4,5646,39.955255,-82.881798,0.0,39.955255,-82.881798,0.0,39.955554,-82.882520,39.957720,-82.967026
...,...,...,...,...,...,...,...,...,...,...,...
4130,13637,39.975582,-83.012655,0.0,39.995013,-83.007298,0.0,39.978411,-83.003898,39.995886,-83.007426
4131,13642,40.008562,-82.935728,0.0,39.969387,-83.004262,0.0,40.010357,-82.947846,39.968768,-83.003008
4132,13643,40.128400,-82.993658,1.0,40.024691,-82.932885,0.0,40.060719,-82.909141,40.027414,-82.933207
4133,13647,40.004199,-82.786555,1.0,39.858534,-82.833692,0.0,39.980259,-82.837110,39.860402,-82.825778


In [47]:
non_origin_destin_columns_check=['originaddresslat', 'originaddresslong','destinaddresslat', 'destinaddresslong']
non_prev_next_columns_check=['prevtransferscode','nexttransferscode']
non_origin_destin_columns=check_all_characters_present(cota_df,non_origin_destin_columns_check)
non_prev_next_columns=check_all_characters_present(cota_df,non_prev_next_columns_check)
non_origin_destin_columns.sort()
non_prev_next_columns.sort()
non_origin_destin_columns

['DESTIN_ADDRESS_LAT_',
 'DESTIN_ADDRESS_LONG_',
 'ORIGIN_ADDRESS_LAT_',
 'ORIGIN_ADDRESS_LONG_']

In [48]:
non_prev_next_columns

['NEXT_TRANSFERS_Code_', 'PREV_TRANSFERS_Code_']

In [49]:
ke_df[[*non_prev_next_columns,'KE_PREV_TRANSFERSCode','KE_NEXT_TRANSFERSCode']]=ke_df[[*non_prev_next_columns,'KE_PREV_TRANSFERSCode','KE_NEXT_TRANSFERSCode']].fillna(0)

In [50]:
ke_df[[*non_origin_destin_columns,'KE_ORIGIN_ADDRESS_LAT','KE_ORIGIN_ADDRESS_LONG','KE_DESTIN_ADDRESS_LAT','KE_DESTIN_ADDRESS_LONG']]=ke_df[[*non_origin_destin_columns,'KE_ORIGIN_ADDRESS_LAT','KE_ORIGIN_ADDRESS_LONG','KE_DESTIN_ADDRESS_LAT','KE_DESTIN_ADDRESS_LONG']].fillna(0)

[
'DESTIN_ADDRESS_LAT',0
 'DESTIN_ADDRESS_LONG',1
 'NEXT_TRANSFERSCode',2
 'ORIGIN_ADDRESS_LAT',3
 'ORIGIN_ADDRESS_LONG',4
 'PREV_TRANSFERSCode',5
 'STOP_OFF_LAT',6
 'STOP_OFF_LONG',7
 'STOP_ON_LAT',8
 'STOP_ON_LONG'9
 ]

In [51]:
# for index, row in ke_df.iterrows():
#     num_changes = 0
    
#     if (round(row[origin_destin_prev_next_columns[3]],3)==round(row['KE_ORIGIN_ADDRESS_LAT'],3)) and (round(row[origin_destin_prev_next_columns[4]],3)==round(row['KE_ORIGIN_ADDRESS_LONG'],3)):
#         print(round(row[origin_destin_prev_next_columns[0]],3),round(row['KE_DESTIN_ADDRESS_LAT'],3))
#     else:
#         print("Change in Origin")
#         num_changes+=1
#     if (round(row[origin_destin_prev_next_columns[0]],3)==round(row['KE_DESTIN_ADDRESS_LAT'],3)) and (round(row[origin_destin_prev_next_columns[1]],3)==round(row['KE_DESTIN_ADDRESS_LONG'],3)):
#         print(round(row[origin_destin_prev_next_columns[0]],3),round(row['KE_DESTIN_ADDRESS_LAT'],3))
#         print("Change in DESTIN")
#     else:
#         num_changes+=1
#     if (row[origin_destin_prev_next_columns[2]]!=row['KE_NEXT_TRANSFERSCode']):
#         print(row[origin_destin_prev_next_columns[2]],row['KE_NEXT_TRANSFERSCode'])
#         print(index)
#         print("Change in NEXTTRANSFERCODE")
#         num_changes+=1
#     if (row[origin_destin_prev_next_columns[5]]!=row['KE_PREV_TRANSFERSCode']):
#         print("Change in PREVTRANSFERCODE")
#         num_changes+=1
#     ke_df.loc[index, 'Change Count'] = num_changes


In [52]:
def get_distance_between_coordinates(lat1, lon1, lat2, lon2):
    try:
        lat1 = float(lat1)
        lon1 = float(lon1)
        lat2 = float(lat2)
        lon2 = float(lon2)
        
        coords_1 = (lat1, lon1)
        coords_2 = (lat2, lon2)
        
        distance = geodesic(coords_1, coords_2).miles
        return distance
    except (ValueError, TypeError) as e:
        # Handle the exception here
        pass

In [53]:
columns_to_check_for_route_processing=['routesurveyedcode','have5minforsurvecode', 'intervinit']
columns_to_check_for_route=check_all_characters_present(ke_df,columns_to_check_for_route_processing)
columns_to_check_for_route.sort()
columns_to_check_for_route

['HAVE_5_MIN_FOR_SURVECode', 'INTERV_INIT', 'ROUTE_SURVEYEDCode']

In [54]:
for index, row in ke_df.iterrows():
    num_changes = 0
    elvis_origin_lat=row['KE_ORIGIN_ADDRESS_LAT']
    elvis_origin_long=row['KE_ORIGIN_ADDRESS_LONG']
    non_elvis_origin_lat=row[non_origin_destin_columns[2]]
    non_elvis_origin_long=row[non_origin_destin_columns[3]]
    
    elvis_destin_lat=row['KE_DESTIN_ADDRESS_LAT']
    elvis_destin_long=row['KE_DESTIN_ADDRESS_LONG']
    non_elvis_destin_lat=row[non_origin_destin_columns[0]]
    non_elvis_destin_long=row[non_origin_destin_columns[1]]
    
    if all([elvis_origin_lat, elvis_origin_long, non_elvis_origin_lat, non_elvis_origin_long]):
        distance=get_distance_between_coordinates(elvis_origin_lat,elvis_origin_long,non_elvis_origin_lat,non_elvis_origin_long)
        if distance > 0.15:
            ke_df.loc[index,'Origin Change']=1
            num_changes += 1
        else:
            ke_df.loc[index,'Origin Change']=0
    elif elvis_origin_lat and elvis_origin_long:
        num_changes += 1
        ke_df.loc[index,'Origin Change']=1
    elif non_elvis_origin_lat and non_elvis_origin_long:
        num_changes += 1
        ke_df.loc[index,'Origin Change']=1
    else:
        ke_df.loc[index,'Origin Change']=0
            

    if all([elvis_destin_lat, elvis_destin_long, non_elvis_destin_lat, non_elvis_destin_long]):
        distance=get_distance_between_coordinates(elvis_destin_lat,elvis_destin_long,non_elvis_destin_lat,non_elvis_destin_long)
        if distance > 0.15:
            ke_df.loc[index,'Destin Change']=1
            num_changes += 1
        else:
            ke_df.loc[index,'Destin Change']=0
    elif elvis_destin_lat and elvis_destin_long:
        ke_df.loc[index,'Destin Change']=1
        num_changes += 1
    elif non_elvis_destin_lat and non_elvis_destin_long:
        ke_df.loc[index,'Destin Change']=1
        num_changes += 1
    else:
        ke_df.loc[index,'Destin Change']=0

    if row[non_prev_next_columns[0]] == row['KE_NEXT_TRANSFERSCode']:
        ke_df.loc[index,'NextTrans Change']=0
    else:
        ke_df.loc[index,'NextTrans Change']=1
        num_changes += 1

    if row[non_prev_next_columns[1]] == row['KE_PREV_TRANSFERSCode']:
        ke_df.loc[index,'PrevTrans Change']=0

    else:
        ke_df.loc[index,'PrevTrans Change']=1
        num_changes += 1

    ke_df.at[index, 'Change Count'] = num_changes


In [55]:
ke_df['Low Changes'] = np.where(ke_df['Change Count'] <= 1, 'Yes', 'No')

In [56]:
ke_df.head(10)

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,id,Completed,INTERV_INIT,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,HAVE_5_MIN_FOR_SURVECode,HAVE_5_MIN_FOR_SURVE,ORIGIN_PLACE_TYPE,ORIGIN_TRANSPORTCode,ORIGIN_TRANSPORT,DESTIN_PLACE_TYPE,DESTIN_TRANSPORTCode,DESTIN_TRANSPORT,_INTRV_NOTE,ELVIS_STATUS,ELVIS_COMMENT,ROUTE_STATUS,Stops_Status,Test_Status,Traditional_Check,...,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1,KE_DESTIN_ADDRESS_LAT,KE_DESTIN_ADDRESS_LONG,KE_NEXT_TRANSFERSCode,KE_ORIGIN_ADDRESS_LAT,KE_ORIGIN_ADDRESS_LONG,KE_PREV_TRANSFERSCode,KE_STOP_OFF_LAT,KE_STOP_OFF_LONG,KE_STOP_ON_LAT,KE_STOP_ON_LONG,DESTIN_ADDRESS_LAT_,DESTIN_ADDRESS_LONG_,NEXT_TRANSFERS_Code_,ORIGIN_ADDRESS_LAT_,ORIGIN_ADDRESS_LONG_,PREV_TRANSFERS_Code_,STOP_OFF_LAT_,STOP_OFF_LONG_,STOP_ON_LAT_,STOP_ON_LONG_,Origin Change,Destin Change,NextTrans Change,PrevTrans Change,Change Count,Low Changes
0,2023-11-03,5632,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5632,2023-10-30 12:24:56,JTC,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",Your usual workplace,5,Dropped off by someone going elsewhere,Your HOME,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,...,0.0,NaN,NaN,NaN,39.941384,-82.853440,0.0,40.014357,-82.986368,1.0,39.943327,-82.853257,39.962784,-83.000898,39.941384,-82.853440,0.0,40.014357,-82.986368,1.0,39.943327,-82.853257,39.962784,-83.000898,0.0,0.0,0.0,0.0,0.0,Yes
1,2023-11-03,5634,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5634,2023-10-30 12:30:07,BLN,COT_1_2_00,2 E MAIN/N HIGH - SOUTH/EAST,1,"<span style=""color:#000080;""><span style=""font...",Your HOME,1,Walk,Grocery / Food Shopping,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,...,0.0,NaN,NaN,NaN,39.889314,-83.001389,1.0,39.963826,-82.999827,0.0,39.956175,-82.998467,39.962784,-83.000898,39.889314,-83.001389,1.0,39.963826,-82.999827,0.0,39.956175,-82.998467,39.962784,-83.000898,0.0,0.0,0.0,0.0,0.0,Yes
2,2023-11-03,5636,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5636,2023-10-30 12:36:28,JTC,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",Social Visit (friends/relatives),1,Walk,Your HOME,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,...,0.0,NaN,NaN,NaN,39.944775,-82.877817,0.0,39.989274,-83.005284,0.0,39.944646,-82.876777,39.988958,-83.005951,39.944775,-82.877817,0.0,39.989274,-83.005284,0.0,39.944646,-82.876777,39.988958,-83.005951,0.0,0.0,0.0,0.0,0.0,Yes
3,2023-11-03,5639,HereAPI,"T A Divya, Brian, Regina",Remove,Round trip,,Elvis_Review,Elvis_Review,NaN,5639,2023-10-30 12:46:45,MCB,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",College / University (students only),1,Walk,Your HOME,1,Walk,NaN,Delete,NaN,Review,Review,Tested,0,...,0.0,NaN,NaN,NaN,39.968780,-82.986813,0.0,39.968780,-82.986813,2.0,39.965149,-83.001318,39.968203,-83.001929,39.968780,-82.986813,0.0,39.968780,-82.986813,2.0,39.965149,-83.001318,39.968203,-83.001929,0.0,0.0,0.0,0.0,0.0,Yes
4,2023-11-03,5646,HereAPI,"T A Divya, Brian, Regina",Use,NaN,,Elvis_Review,Elvis_Review,"o= home = E Main St & Wilson Ave, Columbus, OH...",5646,2023-10-30 12:44:33,CDK,COT_1_2_00,2 E MAIN/N HIGH - SOUTH/EAST,1,"<span style=""color:#000080;""><span style=""font...","Personal Business (bank, haircut, post office)",1,Walk,"Personal Business (bank, haircut, post office)",1,Walk,NaN,Fixable - needs additional review,o= home,Review,Review,Tested,0,...,0.0,NaN,NaN,NaN,39.955255,-82.881798,0.0,39.955255,-82.881798,0.0,39.955554,-82.882520,39.957720,-82.967026,39.955255,-82.881798,0.0,39.955255,-82.881798,0.0,39.955554,-82.882520,39.957720,-82.967026,0.0,0.0,0.0,0.0,0.0,Yes
5,2023-11-03,5647,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5647,2023-10-30 12:54:40,JTC,COT_1_1_00,1 KENNY/LIVINGSTON - EAST

In [57]:
stop_on_off_columns_check=['stoponlat', 'stoponlong', 'stopofflat', 'stopofflong']
stop_on_off_columns=check_all_characters_present(ke_df,stop_on_off_columns_check)
stop_on_off_columns.sort()
stop_on_off_columns

['STOP_OFF_LAT_', 'STOP_OFF_LONG_', 'STOP_ON_LAT_', 'STOP_ON_LONG_']

In [58]:
ke_df[[*stop_on_off_columns,'KE_STOP_OFF_LAT', 'KE_STOP_OFF_LONG', 'KE_STOP_ON_LAT', 'KE_STOP_ON_LONG']]=ke_df[[*stop_on_off_columns,'KE_STOP_OFF_LAT', 'KE_STOP_OFF_LONG', 'KE_STOP_ON_LAT', 'KE_STOP_ON_LONG']].fillna(0)

In [59]:
ke_df.head()

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,id,Completed,INTERV_INIT,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,HAVE_5_MIN_FOR_SURVECode,HAVE_5_MIN_FOR_SURVE,ORIGIN_PLACE_TYPE,ORIGIN_TRANSPORTCode,ORIGIN_TRANSPORT,DESTIN_PLACE_TYPE,DESTIN_TRANSPORTCode,DESTIN_TRANSPORT,_INTRV_NOTE,ELVIS_STATUS,ELVIS_COMMENT,ROUTE_STATUS,Stops_Status,Test_Status,Traditional_Check,...,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1,KE_DESTIN_ADDRESS_LAT,KE_DESTIN_ADDRESS_LONG,KE_NEXT_TRANSFERSCode,KE_ORIGIN_ADDRESS_LAT,KE_ORIGIN_ADDRESS_LONG,KE_PREV_TRANSFERSCode,KE_STOP_OFF_LAT,KE_STOP_OFF_LONG,KE_STOP_ON_LAT,KE_STOP_ON_LONG,DESTIN_ADDRESS_LAT_,DESTIN_ADDRESS_LONG_,NEXT_TRANSFERS_Code_,ORIGIN_ADDRESS_LAT_,ORIGIN_ADDRESS_LONG_,PREV_TRANSFERS_Code_,STOP_OFF_LAT_,STOP_OFF_LONG_,STOP_ON_LAT_,STOP_ON_LONG_,Origin Change,Destin Change,NextTrans Change,PrevTrans Change,Change Count,Low Changes
0,2023-11-03,5632,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5632,2023-10-30 12:24:56,JTC,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",Your usual workplace,5,Dropped off by someone going elsewhere,Your HOME,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,...,0.0,NaN,NaN,NaN,39.941384,-82.853440,0.0,40.014357,-82.986368,1.0,39.943327,-82.853257,39.962784,-83.000898,39.941384,-82.853440,0.0,40.014357,-82.986368,1.0,39.943327,-82.853257,39.962784,-83.000898,0.0,0.0,0.0,0.0,0.0,Yes
1,2023-11-03,5634,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5634,2023-10-30 12:30:07,BLN,COT_1_2_00,2 E MAIN/N HIGH - SOUTH/EAST,1,"<span style=""color:#000080;""><span style=""font...",Your HOME,1,Walk,Grocery / Food Shopping,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,...,0.0,NaN,NaN,NaN,39.889314,-83.001389,1.0,39.963826,-82.999827,0.0,39.956175,-82.998467,39.962784,-83.000898,39.889314,-83.001389,1.0,39.963826,-82.999827,0.0,39.956175,-82.998467,39.962784,-83.000898,0.0,0.0,0.0,0.0,0.0,Yes
2,2023-11-03,5636,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5636,2023-10-30 12:36:28,JTC,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",Social Visit (friends/relatives),1,Walk,Your HOME,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,...,0.0,NaN,NaN,NaN,39.944775,-82.877817,0.0,39.989274,-83.005284,0.0,39.944646,-82.876777,39.988958,-83.005951,39.944775,-82.877817,0.0,39.989274,-83.005284,0.0,39.944646,-82.876777,39.988958,-83.005951,0.0,0.0,0.0,0.0,0.0,Yes
3,2023-11-03,5639,HereAPI,"T A Divya, Brian, Regina",Remove,Round trip,,Elvis_Review,Elvis_Review,NaN,5639,2023-10-30 12:46:45,MCB,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",College / University (students only),1,Walk,Your HOME,1,Walk,NaN,Delete,NaN,Review,Review,Tested,0,...,0.0,NaN,NaN,NaN,39.968780,-82.986813,0.0,39.968780,-82.986813,2.0,39.965149,-83.001318,39.968203,-83.001929,39.968780,-82.986813,0.0,39.968780,-82.986813,2.0,39.965149,-83.001318,39.968203,-83.001929,0.0,0.0,0.0,0.0,0.0,Yes
4,2023-11-03,5646,HereAPI,"T A Divya, Brian, Regina",Use,NaN,,Elvis_Review,Elvis_Review,"o= home = E Main St & Wilson Ave, Columbus, OH...",5646,2023-10-30 12:44:33,CDK,COT_1_2_00,2 E MAIN/N HIGH - SOUTH/EAST,1,"<span style=""color:#000080;""><span style=""font...","Personal Business (bank, haircut, post office)",1,Walk,"Personal Business (bank, haircut, post office)",1,Walk,NaN,Fixable - needs additional review,o= home,Review,Review,Tested,0,...,0.0,NaN,NaN,NaN,39.955255,-82.881798,0.0,39.955255,-82.881798,0.0,39.955554,-82.882520,39.957720,-82.967026,39.955255,-82.881798,0.0,39.955255,-82.881798,0.0,39.955554,-82.882520,39.957720,-82.967026,0.0,0.0,0.0,0.0,0.0,Yes


In [60]:
ke_stop_on_off_column_check=['kestopofflat','kestopofflong','kestoponlat','kestoponlong']
ke_stop_on_off_column=check_all_characters_present(ke_df,ke_stop_on_off_column_check)
ke_stop_on_off_column.sort()
ke_stop_on_off_column

['KE_STOP_OFF_LAT', 'KE_STOP_OFF_LONG', 'KE_STOP_ON_LAT', 'KE_STOP_ON_LONG']

In [61]:
for index, row in ke_df.iterrows():
    if (
        (round(row[stop_on_off_columns[0]], 3) == round(row[ke_stop_on_off_column[0]], 3)) and
        (round(row[stop_on_off_columns[1]], 3) == round(row[ke_stop_on_off_column[1]], 3))
    ) and (
        (round(row[stop_on_off_columns[2]], 3) == round(row[ke_stop_on_off_column[2]], 3)) and
        (round(row[stop_on_off_columns[3]], 3) == round(row[ke_stop_on_off_column[3]], 3))
    ):
        ke_df.loc[index, 'Route Changes'] = 0
    else:
        ke_df.loc[index, 'Route Changes'] = 1


In [62]:
# ke_df[['KE_ORIGIN_ADDRESS_LAT','KE_ORIGIN_ADDRESS_LONG','ORIGIN_ADDRESS_LAT','ORIGIN_ADDRESS_LONG','Origin Change']][ke_df['Origin Change']==1]

In [63]:
# for index, row in ke_df.iterrows():
#     num_changes = 0
    
    
#     if (round(row[origin_destin_prev_next_columns[3]], 3) == round(row['KE_ORIGIN_ADDRESS_LAT'], 3)) and \
#        (round(row[origin_destin_prev_next_columns[4]], 3) == round(row['KE_ORIGIN_ADDRESS_LONG'], 3)):
#         print(f"Origin coordinates unchanged: {round(row[origin_destin_prev_next_columns[3]], 3)}, {round(row[origin_destin_prev_next_columns[4]], 3)},{round(row['KE_ORIGIN_ADDRESS_LAT'], 3)},{round(row['KE_ORIGIN_ADDRESS_LONG'], 3)}")
#     else:
#         print(f"Origin coordinates changed: {round(row[origin_destin_prev_next_columns[3]], 3)},{round(row[origin_destin_prev_next_columns[4]], 3)},{round(row['KE_ORIGIN_ADDRESS_LAT'], 3)},{round(row['KE_ORIGIN_ADDRESS_LONG'], 3)}")
#         print("Change in Origin")
#         num_changes += 1

#     if (round(row[origin_destin_prev_next_columns[0]], 3) == round(row['KE_DESTIN_ADDRESS_LAT'], 3)) and \
#        (round(row[origin_destin_prev_next_columns[1]], 3) == round(row['KE_DESTIN_ADDRESS_LONG'], 3)):
#         print(f"Destination coordinates unchanged: {round(row[origin_destin_prev_next_columns[0]], 3)},{round(row[origin_destin_prev_next_columns[1]], 3)},{round(row['KE_DESTIN_ADDRESS_LAT'], 3)},{round(row['KE_DESTIN_ADDRESS_LONG'], 3)}")
#     else:
#         print(f"Destination coordinates Changed: {round(row[origin_destin_prev_next_columns[0]], 3)},{round(row[origin_destin_prev_next_columns[1]], 3)},{round(row['KE_DESTIN_ADDRESS_LAT'], 3)},{round(row['KE_DESTIN_ADDRESS_LONG'], 3)}")
#         print("Change in DESTIN")
#         num_changes += 1

#     if row[origin_destin_prev_next_columns[2]] == row['KE_NEXT_TRANSFERSCode']:
#         print(f"UnChange in NEXTTRANSFERCODE: {row[origin_destin_prev_next_columns[2]]}, {row['KE_NEXT_TRANSFERSCode']}")
#         print(index)
#     else:
#         print(f"Change in NEXTTRANSFERCODE: {row[origin_destin_prev_next_columns[2]]}, {row['KE_NEXT_TRANSFERSCode']}")
#         num_changes += 1

#     if row[origin_destin_prev_next_columns[5]] == row['KE_PREV_TRANSFERSCode']:
        
#         print("UnChange in PREVTRANSFERCODE")
#     else:
#         print(f"Change in PREVTRANSFERCODE: {row[origin_destin_prev_next_columns[5]]}, {row['KE_PREV_TRANSFERSCode']}")
#         num_changes += 1

#     ke_df.at[index, 'Change Count'] = num_changes


In [64]:
ke_df.shape

(4135, 70)

In [65]:
summary_df=pd.DataFrame()

In [66]:
route_surveyed_column_check=['routesurveyedcode']
route_surveyed_column=check_all_characters_present(ke_df,route_surveyed_column_check)
route_surveyed_column

['ROUTE_SURVEYEDCode']

In [67]:
def route_splited(value):
    return '_'.join(value.split('_')[:-1])

In [68]:
ke_df['ROUTE_SURVEYEDCode_Splited']=ke_df[route_surveyed_column[0]].apply(route_splited)

In [69]:
# ke_df[['ROUTE_SURVEYEDCode_Splited']]

In [70]:
route_surveyed_column_splited_check=['routesurveyedcodesplited']
route_surveyed_column_splited=check_all_characters_present(ke_df,route_surveyed_column_splited_check)
route_surveyed_column_splited

['ROUTE_SURVEYEDCode_Splited']

In [71]:
route_values=ke_df[route_surveyed_column_splited[0]].unique()

In [72]:
len(route_values)

34

In [73]:
summary_df['Route'] = route_values

In [74]:
for index, row in summary_df.iterrows():
    overall_reviews=ke_df.shape[0]
    
    route_surveyed_value = row['Route']  # Assuming 'Route' is the correct column name in summary_df
    review_filter_condition = (
        (ke_df[route_surveyed_column_splited[0]] == route_surveyed_value) &
        ((ke_df['Final_Usage'].str.lower() == 'use') | (ke_df['Final_Usage'].str.lower() == 'remove'))
    )
    total_reviews = ke_df[review_filter_condition].shape[0]  # Adjust as needed
    removal_filter_condition = (
        (ke_df[route_surveyed_column_splited[0]] == route_surveyed_value) &
        (ke_df['Final_Usage'].str.lower() == 'remove')
    )
    total_removals = ke_df[removal_filter_condition].shape[0]
    removal_survey_ids_filter_condition=(
        (ke_df[route_surveyed_column_splited[0]]==route_surveyed_value)&
        (ke_df['Final_Usage'].str.lower()=='remove')
    )
    sum_of_origin_change_filter=(
            (ke_df[route_surveyed_column_splited[0]]==route_surveyed_value)&
            (ke_df['Origin Change']==1)
    )
    sum_of_destin_change_filter=(
            (ke_df[route_surveyed_column_splited[0]]==route_surveyed_value)&
            (ke_df['Destin Change']==1)
    )
    
    sum_of_prev_change_filter=(
            (ke_df[route_surveyed_column_splited[0]]==route_surveyed_value)&
            (ke_df['PrevTrans Change']==1)
    )
    
    sum_of_next_change_filter=(
            (ke_df[route_surveyed_column_splited[0]]==route_surveyed_value)&
            (ke_df['NextTrans Change']==1)
    )
    sum_of_record_change_filter=(
            (ke_df[route_surveyed_column_splited[0]]==route_surveyed_value)&
            (ke_df['Change Count']!=0)
    )
    removal_survey_ids=ke_df['id'][removal_survey_ids_filter_condition].values
    summary_df.loc[index, 'Total_Reviews'] = total_reviews
    summary_df.loc[index, 'Total_Removals'] = total_removals
    sum_origin_change=ke_df[sum_of_origin_change_filter].shape[0]
    sum_destin_change=ke_df[sum_of_destin_change_filter].shape[0]
    sum_nexttrans_change=ke_df[sum_of_next_change_filter].shape[0]
    sum_prevtrans_change=ke_df[sum_of_prev_change_filter].shape[0]
    sum_record_change=ke_df[sum_of_record_change_filter].shape[0]
    if total_reviews:
        summary_df.loc[index,'Removal_Rate_Percentage']=(total_removals*100)/total_reviews
    else:
        summary_df.loc[index,'Removal_Rate_Percentage']=0
    if overall_reviews:
        summary_df.loc[index,'Route_Reviewed_Percentage']=(total_reviews*100)/overall_reviews
    else:
        summary_df.loc[index,'Route_Reviewed_Percentage']=0
    summary_df.loc[index, 'Removed_Survey_ids'] = ', '.join(map(str, removal_survey_ids))
    summary_df.loc[index,'Sum of Origin Change']=sum_origin_change
    summary_df.loc[index,'Sum of Destin Change']=sum_destin_change
    summary_df.loc[index,'Sum of NextTrans Change']=sum_nexttrans_change
    summary_df.loc[index,'Sum of PrevTrans Change']=sum_prevtrans_change
    summary_df.loc[index,'Sum of Record Change']=sum_record_change
    summary_df.loc[index,'Origin Change Percentage']=f"{round((sum_origin_change*100)/total_reviews,2)}%"
    summary_df.loc[index,'Destin Change Percentage']=f"{round((sum_destin_change*100)/total_reviews,2)}%"
    summary_df.loc[index,'NextTrans Change Percentage']=f"{round((sum_nexttrans_change*100)/total_reviews,2)}%"
    summary_df.loc[index,'PrevTrans Change Percentage']=f"{round((sum_prevtrans_change*100)/total_reviews,2)}%"
    summary_df.loc[index,'Record Change Percentage']=f"{round((sum_record_change*100)/total_reviews,2)}%"



In [75]:
summary_df

,Route,Total_Reviews,Total_Removals,Removal_Rate_Percentage,Route_Reviewed_Percentage,Removed_Survey_ids,Sum of Origin Change,Sum of Destin Change,Sum of NextTrans Change,Sum of PrevTrans Change,Sum of Record Change,Origin Change Percentage,Destin Change Percentage,NextTrans Change Percentage,PrevTrans Change Percentage,Record Change Percentage
0,COT_1_1,626.0,17.0,2.715655,15.139057,"5639, 5786, 5847, 6744, 7171, 7239, 7305, 7698...",2.0,1.0,48.0,39.0,86.0,0.32%,0.16%,7.67%,6.23%,13.74%
1,COT_1_2,663.0,30.0,4.524887,16.033857,"7114, 7537, 7568, 7632, 7638, 8230, 8316, 8329...",2.0,1.0,65.0,68.0,113.0,0.3%,0.15%,9.8%,10.26%,17.04%
2,COT_1_22,158.0,11.0,6.962025,3.821040,"5959, 9810, 11004, 11043, 11199, 11232, 11263,...",0.0,0.0,19.0,19.0,34.0,0.0%,0.0%,12.03%,12.03%,21.52%
3,COT_1_10,615.0,33.0,5.365854,14.873035,"6528, 6894, 7704, 7815, 7821, 7936, 8637, 8695...",3.0,1.0,57.0,69.0,120.0,0.49%,0.16%,9.27%,11.22%,19.51%
4,COT_1_11,46.0,12.0,26.086957,1.112455,"6111, 6137, 6143, 10975, 11016, 11018, 11112, ...",0.0,0.0,3.0,4.0,6.0,0.0%,0.0%,6.52%,8.7%,13.04%
5,COT_1_8,458.0,33.0,7.205240,11.076179,"8487, 8498, 8515, 8666, 8797, 8836, 8860, 8959...",4.0,1.0,27.0,29.0,57.0,0.87%,0.22%,5.9%,6.33%,12.45%
6,COT_1_CMAX,424.0,17.0,4.009434,10.253930,"6787, 7131, 7152, 7258, 7294, 7656, 8004, 8141...",3.0,3.0,48.0,41.0,84.0,0.71%,0.71%,11.32%,9.67%,19.81%
7,COT_1_5,231.0,17.0,7.359307,5.586457,"6792, 7130, 7384, 9238, 9327, 9334, 9338, 9354...",0.0,0.0,31.0,25.0,53.0,0.0%,0.0%,13.42%,10.82%,22.94%
8,COT_1_ZOO,2.0,1.0,50.000000,0.048368,7027,0.0,0.0,1.0,1.0,1.0,0.0%,0.0%,50.0%,50.0%,50.0%
9,COT_1_43,2.0,0.0,0.000000,0.048368,,0.0,0.0,0.0,0.0,0.0,0.0%,0.0%,0.0%,0.0%,0.0%


In [76]:
final_reviewer_column_check=['finalreviewer']
final_reviewer_column=check_all_characters_present(ke_df,final_reviewer_column_check)
final_reviewer_column

['FINAL_REVIEWER']

In [77]:
def unique_reviewers_values(value):
    names=[name.strip() for name in str(value).replace('.', ',').split(',')]
    return names[0] 

ke_df['FINAL_REVIEWER_Splited']=ke_df[final_reviewer_column[0]].apply(unique_reviewers_values)

In [78]:
ke_df[['FINAL_REVIEWER_Splited']].tail()

,FINAL_REVIEWER_Splited
4130,Divya sravani
4131,Divya sravani
4132,Divya sravani
4133,Divya sravani
4134,Divya sravani


In [79]:
reviewer_values=ke_df['FINAL_REVIEWER_Splited'].unique()

In [80]:
reviewer_df=pd.DataFrame()

In [81]:
reviewer_df['Row Labels']=reviewer_values

In [82]:
for index,row in reviewer_df.iterrows():
    reviewer_reviewed_value = row['Row Labels']  # Assuming 'Route' is the correct column name in summary_df
    review_filter_condition = (
        (ke_df['FINAL_REVIEWER_Splited'] == reviewer_reviewed_value) &
        ((ke_df['Final_Usage'].str.lower() == 'use') | (ke_df['Final_Usage'].str.lower() == 'remove'))
    )
    origin_change_filter_condition = (
        (ke_df['FINAL_REVIEWER_Splited'] == reviewer_reviewed_value) &
        (ke_df['Origin Change']==1)
    )
    destin_change_filter_condition = (
        (ke_df['FINAL_REVIEWER_Splited'] == reviewer_reviewed_value) &
        (ke_df['Destin Change']==1)
    )
    sum_of_prev_change_filter=(
        (ke_df['FINAL_REVIEWER_Splited'] == reviewer_reviewed_value)&
        (ke_df['PrevTrans Change']==1)
    )
    
    sum_of_next_change_filter=(
        (ke_df['FINAL_REVIEWER_Splited'] == reviewer_reviewed_value)&
        (ke_df['NextTrans Change']==1)
    )
    sum_of_record_change_filter=(
        (ke_df['FINAL_REVIEWER_Splited'] == reviewer_reviewed_value)&
        (ke_df['Change Count']!=0)
    )
    count_elvis_id = ke_df[review_filter_condition].shape[0]  
    origin_change_sum = ke_df[origin_change_filter_condition].shape[0]  
    destin_change_sum = ke_df[destin_change_filter_condition].shape[0]  
    prevtrans_change_sum = ke_df[sum_of_prev_change_filter].shape[0]  
    nexttrans_change_sum = ke_df[sum_of_next_change_filter].shape[0]  
    record_change_sum = ke_df[sum_of_record_change_filter].shape[0]  
    reviewer_df.loc[index,'Count of Elvis_ID']=count_elvis_id
    reviewer_df.loc[index,'Sum of Origin Change']=origin_change_sum
    reviewer_df.loc[index,'Sum of Destin Change']=destin_change_sum
    reviewer_df.loc[index,'Sum of PrevTrans Change']=prevtrans_change_sum
    reviewer_df.loc[index,'Sum of NextTrans Change']=nexttrans_change_sum
    reviewer_df.loc[index,'Sum of Record Change']=record_change_sum
    reviewer_df.loc[index,'Origin Change Percentage']=f"{round((origin_change_sum*100)/count_elvis_id,2)}%"
    reviewer_df.loc[index,'Destin Change Percentage']=f"{round((destin_change_sum*100)/count_elvis_id,2)}%"
    reviewer_df.loc[index,'NextTrans Change Percentage']=f"{round((nexttrans_change_sum*100)/count_elvis_id,2)}%"
    reviewer_df.loc[index,'PrevTrans Change Percentage']=f"{round((prevtrans_change_sum*100)/count_elvis_id,2)}%"
    reviewer_df.loc[index,'Record Change Percentage']=f"{round((record_change_sum*100)/count_elvis_id,2)}%"
    
reviewer_df

,Row Labels,Count of Elvis_ID,Sum of Origin Change,Sum of Destin Change,Sum of PrevTrans Change,Sum of NextTrans Change,Sum of Record Change,Origin Change Percentage,Destin Change Percentage,NextTrans Change Percentage,PrevTrans Change Percentage,Record Change Percentage
0,T A Divya,178.0,0.0,0.0,13.0,14.0,26.0,0.0%,0.0%,7.87%,7.3%,14.61%
1,Sneh,395.0,1.0,0.0,21.0,38.0,55.0,0.25%,0.0%,9.62%,5.32%,13.92%
2,Rutwa,72.0,0.0,0.0,0.0,0.0,0.0,0.0%,0.0%,0.0%,0.0%,0.0%
3,Sameera,235.0,3.0,0.0,33.0,24.0,53.0,1.28%,0.0%,10.21%,14.04%,22.55%
4,Kesar,284.0,0.0,1.0,32.0,22.0,47.0,0.0%,0.35%,7.75%,11.27%,16.55%
5,Karthik,125.0,2.0,1.0,2.0,5.0,10.0,1.6%,0.8%,4.0%,1.6%,8.0%
6,Priyanka,99.0,0.0,0.0,3.0,1.0,4.0,0.0%,0.0%,1.01%,3.03%,4.04%
7,Saurav,230.0,2.0,0.0,16.0,10.0,26.0,0.87%,0.0%,4.35%,6.96%,11.3%
8,Divya sravani,400.0,1.0,0.0,58.0,46.0,89.0,0.25%,0.0%,11.5%,14.5%,22.25%
9,Kalpesh,139.0,2.0,1.0,17.0,20.0,36.0,1.44%,0.72%,14.39%,12.23%,25.9%


In [83]:
pd.concat([summary_df,reviewer_df],axis=1)

,Route,Total_Reviews,Total_Removals,Removal_Rate_Percentage,Route_Reviewed_Percentage,Removed_Survey_ids,Sum of Origin Change,Sum of Destin Change,Sum of NextTrans Change,Sum of PrevTrans Change,Sum of Record Change,Origin Change Percentage,Destin Change Percentage,NextTrans Change Percentage,PrevTrans Change Percentage,Record Change Percentage,Row Labels,Count of Elvis_ID,Sum of Origin Change,Sum of Destin Change,Sum of PrevTrans Change,Sum of NextTrans Change,Sum of Record Change,Origin Change Percentage,Destin Change Percentage,NextTrans Change Percentage,PrevTrans Change Percentage,Record Change Percentage
0,COT_1_1,626.0,17.0,2.715655,15.139057,"5639, 5786, 5847, 6744, 7171, 7239, 7305, 7698...",2.0,1.0,48.0,39.0,86.0,0.32%,0.16%,7.67%,6.23%,13.74%,T A Divya,178.0,0.0,0.0,13.0,14.0,26.0,0.0%,0.0%,7.87%,7.3%,14.61%
1,COT_1_2,663.0,30.0,4.524887,16.033857,"7114, 7537, 7568, 7632, 7638, 8230, 8316, 8329...",2.0,1.0,65.0,68.0,113.0,0.3%,0.15%,9.8%,10.26%,17.04%,Sneh,395.0,1.0,0.0,21.0,38.0,55.0,0.25%,0.0%,9.62%,5.32%,13.92%
2,COT_1_22,158.0,11.0,6.962025,3.821040,"5959, 9810, 11004, 11043, 11199, 11232, 11263,...",0.0,0.0,19.0,19.0,34.0,0.0%,0.0%,12.03%,12.03%,21.52%,Rutwa,72.0,0.0,0.0,0.0,0.0,0.0,0.0%,0.0%,0.0%,0.0%,0.0%
3,COT_1_10,615.0,33.0,5.365854,14.873035,"6528, 6894, 7704, 7815, 7821, 7936, 8637, 8695...",3.0,1.0,57.0,69.0,120.0,0.49%,0.16%,9.27%,11.22%,19.51%,Sameera,235.0,3.0,0.0,33.0,24.0,53.0,1.28%,0.0%,10.21%,14.04%,22.55%
4,COT_1_11,46.0,12.0,26.086957,1.112455,"6111, 6137, 6143, 10975, 11016, 11018, 11112, ...",0.0,0.0,3.0,4.0,6.0,0.0%,0.0%,6.52%,8.7%,13.04%,Kesar,284.0,0.0,1.0,32.0,22.0,47.0,0.0%,0.35%,7.75%,11.27%,16.55%
5,COT_1_8,458.0,33.0,7.205240,11.076179,"8487, 8498, 8515, 8666, 8797, 8836, 8860, 8959...",4.0,1.0,27.0,29.0,57.0,0.87%,0.22%,5.9%,6.33%,12.45%,Karthik,125.0,2.0,1.0,2.0,5.0,10.0,1.6%,0.8%,4.0%,1.6%,8.0%
6,COT_1_CMAX,424.0,17.0,4.009434,10.253930,"6787, 7131, 7152, 7258, 7294, 7656, 8004, 8141...",3.0,3.0,48.0,41.0,84.0,0.71%,0.71%,11.32%,9.67%,19.81%,Priyanka,99.0,0.0,0.0,3.0,1.0,4.0,0.0%,0.0%,1.01%,3.03%,4.04%
7,COT_1_5,231.0,17.0,7.359307,5.586457,"6792, 7130, 7384, 9238, 9327, 9334, 9338, 9354...",0.0,0.0,31.0,25.0,53.0,0.0%,0.0%,13.42%,10.82%,22.94%,Saurav,230.0,2.0,0.0,16.0,10.0,26.0,0.87%,0.0%,4.35%,6.96%,11.3%
8,COT_1_ZOO,2.0,1.0,50.000000,0.048368,7027,0.0,0.0,1.0,1.0,1.0,0.0%,0.0%,50.0%,50.0%,50.0%,Divya sravani,400.0,1.0,0.0,58.0,46.0,89.0,0.25%,0.0%,11.5%,14.5%,22.25%
9,COT_1_43,2.0,0.0,0.000000,0.048368,,0.0,0.0,0.0,0.0,0.0,0.0%,0.0%,0.0%,0.0%,0.0%,Kalpesh,139.0,2.0,1.0,17.0,20.0,36.0,1.44%,0.72%,14.39%,12.23%,25.9%


# Reviewer Level Information

In [84]:
ke_df=pd.read_excel("COTA_KINGElvis.xlsx",sheet_name='Elvis_Review')
ke_df=ke_df[ke_df['INTERV_INIT']!=999]
ke_df=ke_df[ke_df['HAVE_5_MIN_FOR_SURVECode']==1]
ke_df

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,id,Completed,INTERV_INIT,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,HAVE_5_MIN_FOR_SURVECode,HAVE_5_MIN_FOR_SURVE,ORIGIN_PLACE_TYPE,ORIGIN_TRANSPORTCode,ORIGIN_TRANSPORT,DESTIN_PLACE_TYPE,DESTIN_TRANSPORTCode,DESTIN_TRANSPORT,_INTRV_NOTE,ELVIS_STATUS,ELVIS_COMMENT,ROUTE_STATUS,Stops_Status,Test_Status,Traditional_Check,OD_Distance_Check,Transfer_Distance_Check,2X_REVIEW_CHECK,2X_REVIEW_CHECK.1,2x_REVIEWED_BY,2x_REVIEWED_FLAG,ADMIN_APPROVED,SURVEY_RECOVERY,SURVEY_RECOVERY_REVIEWED_BY,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1
16,2023-11-03,5632,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5632,2023-10-30 12:24:56,JTC,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",Your usual workplace,5,Dropped off by someone going elsewhere,Your HOME,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
17,2023-11-03,5634,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5634,2023-10-30 12:30:07,BLN,COT_1_2_00,2 E MAIN/N HIGH - SOUTH/EAST,1,"<span style=""color:#000080;""><span style=""font...",Your HOME,1,Walk,Grocery / Food Shopping,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
18,2023-11-03,5636,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,5636,2023-10-30 12:36:28,JTC,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",Social Visit (friends/relatives),1,Walk,Your HOME,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
19,2023-11-03,5639,HereAPI,"T A Divya, Brian, Regina",Remove,Round trip,,Elvis_Review,Elvis_Review,NaN,5639,2023-10-30 12:46:45,MCB,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",College / University (students only),1,Walk,Your HOME,1,Walk,NaN,Delete,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
24,2023-11-03,5646,HereAPI,"T A Divya, Brian, Regina",Use,NaN,,Elvis_Review,Elvis_Review,"o= home = E Main St & Wilson Ave, Columbus, OH...",5646,2023-10-30 12:44:33,CDK,COT_1_2_00,2 E MAIN/N HIGH - SOUTH/EAST,1,"<span style=""color:#000080;""><span style=""font...","Personal Business (bank, haircut, post office)",1,Walk,"Personal Business (bank, haircut, post office)",1,Walk,NaN,Fixable - needs additional review,o= home,Review,Review,Tested,0,1,0,1,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5566,2023-12-03,13637,HereAPI,Divya sravani,Use,NaN,,Elvis_Review,Elvis_Review,NaN,13637,2023-12-01 12:08:29,ANC,COT_1_2_00,2 E MAIN/N HIGH - SOUTH/EAST,1,"<span style=""color:#000080;""><span style=""font...",Other shopping,1,Walk,Social Visit (friends/relatives),1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5568,2023-12-03,13642,HereAPI,Divya sravani,Use,NaN,,Elvis_Review,Elvis_Review,NaN,13642,2023-12-01 15:02:36,ANC,COT_1_9_00,9 W MOUND/BRENTNELL - NORTH,1,"<span style=""color:#000080;""><span style=""font...",Your usual workplace,1,Walk,Your HOME,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5569,2023-12-03,13643,HereAPI,Divya sravani,Use,NaN,,Elvis_Review,Elvis_Review,NaN,13643,2023-12-01 15:33:31,ANC,COT_1_9_00,9 W MOUND/BRENTNELL - NORTH,1,"<span style=""color:#000080;""><span style=""font...",Social Visit (friends/relatives),1,Walk,Your HOME,5,Be picked up by someone,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5573,2023-12-03,13647,HereAPI,Divya sravani,Use,NaN,,Elvis_Review,Elvis_Review,NaN,13647,2023-12-01 16:52:58,ANC,COT_1_25_00,25

In [85]:
def reviewer_level_information(ke_df):
    print(ke_df.shape[0])
    columns_to_check_for_review_processing=['finalreviewer','have5minforsurvecode', 'intervinit']
    columns=check_all_characters_present(ke_df,columns_to_check_for_review_processing)
    columns.sort()
    
    def unique_reviewers_values(value):
        names=[name.strip() for name in str(value).replace('.', ',').split(',')]
        return names[0] 

    ke_df[columns[0]]=ke_df[columns[0]].apply(unique_reviewers_values)
    remove_counts = ke_df[columns[0]][ke_df['Final_Usage'].str.lower() == 'remove'].value_counts()
    print(remove_counts)
    # Step 2: Count how many times each unique reviewer appears in the 'FINAL_REVIEWER' column.
            
    unique_reviewers = ke_df[columns[0]].unique()
    value_counts = ke_df[columns[0]].value_counts()

    # Step 3: Calculate the removal rate for each reviewer.
    removal_rate = {}
    for reviewer in unique_reviewers:
        removed = remove_counts.get(reviewer, 0) # Get the removed count for the reviewer, default to 0 if not found
        total_reviews = value_counts.get(reviewer, 0)
        removal_rate[reviewer] = (removed / total_reviews) * 100 if total_reviews != 0 else 0

    # Building the data for DataFrame
    data = []
    for reviewer in unique_reviewers:
        total_reviews = value_counts.get(reviewer, 0)
        removed = remove_counts.get(reviewer, 0)
        rate = removal_rate.get(reviewer, 0)
        percentage_reviews = (total_reviews / len(ke_df)) * 100
        data.append([reviewer, total_reviews, percentage_reviews, removed, rate])

    data = pd.DataFrame(data, columns=[
        "Reviewer",
        "Total Reviewed",
        "Percentage Reviewed (%)",
        "Total Removed",
        "Removal Rate (%)"
    ])
    # data = data[~data['Reviewer'].isin(['NO 5 MIN/TEST', 'Deleted'])]

    return data
reviewer_information=reviewer_level_information(ke_df)

4135
FINAL_REVIEWER
Divya sravani    24
Bhumika          24
Gowtham          22
Karthik          17
T A Divya        13
Raja             12
Himanshu         11
Kesar            10
Kalpesh           9
Srisamhita        9
Sakshi            9
Trusha            8
Saurav            8
Priyanka          4
Vaishnavi         4
Sameera           4
Sneh              4
muskan            4
Rutwa             2
karthik           1
Name: count, dtype: int64


In [86]:
reviewer_information

,Reviewer,Total Reviewed,Percentage Reviewed (%),Total Removed,Removal Rate (%)
0,T A Divya,178,4.304716,13,7.303371
1,Sneh,395,9.552600,4,1.012658
2,Rutwa,72,1.741233,2,2.777778
3,Sameera,235,5.683192,4,1.702128
4,Kesar,284,6.868198,10,3.521127
5,Karthik,125,3.022975,17,13.600000
6,Priyanka,99,2.394196,4,4.040404
7,Saurav,230,5.562273,8,3.478261
8,Divya sravani,400,9.673519,24,6.000000
9,Kalpesh,139,3.361548,9,6.474820


In [87]:
columns_to_check_for_review_processing=['finalreviewer','have5minforsurvecode', 'intervinit']
have5minforsurve_column_check=['have5minforsurvecode']
have5minforsurve_column=check_all_characters_present(ke_df,have5minforsurve_column_check)
columns=check_all_characters_present(ke_df,columns_to_check_for_review_processing)
columns.sort()

columns,have5minforsurve_column

(['FINAL_REVIEWER', 'HAVE_5_MIN_FOR_SURVECode', 'INTERV_INIT'],
 ['HAVE_5_MIN_FOR_SURVECode'])

In [88]:
remove_counts = ke_df[ke_df['Final_Usage'] == 'Remove'][columns[0]].value_counts()
remove_counts

FINAL_REVIEWER
Divya sravani    24
Bhumika          24
Gowtham          22
Karthik          17
T A Divya        13
Raja             12
Himanshu         11
Kesar            10
Kalpesh           9
Srisamhita        9
Sakshi            9
Trusha            8
Saurav            8
Priyanka          4
Vaishnavi         4
Sameera           4
Sneh              4
muskan            4
Rutwa             2
karthik           1
Name: count, dtype: int64

In [89]:
ke_df[ke_df['Final_Usage'] == 'Remove']

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,id,Completed,INTERV_INIT,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,HAVE_5_MIN_FOR_SURVECode,HAVE_5_MIN_FOR_SURVE,ORIGIN_PLACE_TYPE,ORIGIN_TRANSPORTCode,ORIGIN_TRANSPORT,DESTIN_PLACE_TYPE,DESTIN_TRANSPORTCode,DESTIN_TRANSPORT,_INTRV_NOTE,ELVIS_STATUS,ELVIS_COMMENT,ROUTE_STATUS,Stops_Status,Test_Status,Traditional_Check,OD_Distance_Check,Transfer_Distance_Check,2X_REVIEW_CHECK,2X_REVIEW_CHECK.1,2x_REVIEWED_BY,2x_REVIEWED_FLAG,ADMIN_APPROVED,SURVEY_RECOVERY,SURVEY_RECOVERY_REVIEWED_BY,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1
19,2023-11-03,5639,HereAPI,T A Divya,Remove,Round trip,,Elvis_Review,Elvis_Review,NaN,5639,2023-10-30 12:46:45,MCB,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",College / University (students only),1,Walk,Your HOME,1,Walk,NaN,Delete,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
121,2023-11-03,5786,HereAPI,Rutwa,Remove,o=d,,Elvis_Review,Elvis_Review,NaN,5786,2023-10-31 11:49:00,CDK,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",Grocery / Food Shopping,1,Walk,Your HOME,1,Walk,NaN,Delete,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
167,2023-11-03,5847,HereAPI,Sameera,Remove,Use of transit not necessary,,Elvis_Review,Elvis_Review,NaN,5847,2023-10-31 14:23:54,CDK,COT_1_1_00,1 KENNY/LIVINGSTON - EAST/SOUTH,1,"<span style=""color:#000080;""><span style=""font...",Your usual workplace,1,Walk,Your HOME,1,Walk,NaN,Delete,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
257,2023-11-03,5959,HereAPI,Rutwa,Remove,No Transfer Available,,Elvis_Review,Elvis_Review,NaN,5959,2023-11-01 11:18:16,CDK,COT_1_22_01,22 OSU-RICKENBACKER - NORTHWEST,1,"<span style=""color:#000080;""><span style=""font...",Your HOME,1,Walk,Your usual workplace,1,Walk,NaN,Delete,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
370,2023-11-03,6111,HereAPI,Karthik,Remove,Round trip,,Elvis_Review,Elvis_Review,NaN,6111,2023-11-02 12:31:04,CDK,COT_1_11_00,11 BRYDEN/MAIZE - SOUTHEAST,1,"<span style=""color:#000080;""><span style=""font...","Personal Business (bank, haircut, post office)",1,Walk,Your HOME,1,Walk,NaN,NaN,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5300,2023-12-03,13278,HereAPI,Saurav,Remove,No Transfer available,,Elvis_Review,Elvis_Review,NaN,13278,2023-11-30 07:23:44,DRW,COT_1_102_00,102 POLARIS PKWY/N HIGH - NORTH,1,"<span style=""color:#000080;""><span style=""font...",Your HOME,1,Walk,Your usual workplace,1,Walk,NaN,Delete,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5329,2023-12-03,13310,HereAPI,Sneh,Remove,O=B,,Elvis_Review,Elvis_Review,O=D,13310,2023-11-30 09:14:09,JDC,COT_1_11_00,11 BRYDEN/MAIZE - SOUTHEAST,1,"<span style=""color:#000080;""><span style=""font...",Your usual workplace,1,Walk,Your HOME,1,Walk,NaN,Delete,O=D,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5342,2023-12-03,13332,HereAPI,Sneh,Remove,O=D,,Elvis_Review,Elvis_Review,"BAD O, O=D",13332,2023-11-30 10:08:54,MCB,COT_1_CMAX_00,101 CMAX - NORTH,1,"<span style=""color:#000080;""><span style=""font...",Other business related,1,Walk,Your HOME,1,Walk,NaN,Delete,"BAD O, O=D",Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5412,2023-12-03,13425,HereAPI,muskan,Remove,o=d,,Elvis_Review,Elvis_Review,NaN,13425,2023-11-30 13:32:37,JDC,COT_1_1_01,1 KENNY/LIVINGSTON - WEST/NORTH,1,"<span style=""color:#000080;""><span style=""font...",Your usual workplace,1,Walk,Your HOME,1,Walk,NaN,Delete,NaN,Review,Review,Tested,0,0,0,0,NaT,NaN,NaN,

In [90]:
# def unique_reviewers_values(value):
#     names=[name.strip() for name in value.replace('.', ',').split(',')]
#     return names[0] 

In [91]:
# ke_df['FINAL_REVIEWER']=ke_df['FINAL_REVIEWER'].apply(unique_reviewers_values)

In [92]:
# ke_df